# 02 · A spherical SOM (GeodesicSOM), explored interactively

A `GeodesicSOM` puts its neurons on a geodesic dome, so the map has no borders and every neuron has the same
neighbourhood (apart from the 12 five-neighbour corners). Every neuron holds an attribute vector with as many
values as the data. The `SOMViewer` shows the sphere in a map projection and opens on the *Neighbour distance*
layer: triangles between neurons coloured by the distance between their attribute vectors, so clusters show as
dark basins separated by bright ridges.

Script version: [`examples/02_geodesicsom_interactive.py`](../02_geodesicsom_interactive.py) · Needs: numpy, mtgeodesicdome, matplotlib, ipywidgets
(optional: `ipympl` for live figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicsom importable from a checkout; helpers for the controls below

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import numpy as np

from mt.geodesicsom import datasets
from mt.geodesicsom.GeodesicSOM import GeodesicSOM
from mt.geodesicsom.gui import SOMViewer

## 1. Data

In [ ]:
DATA = 'clusters'          # 'clusters', 'iris', 'penguins', 'wine', 'animals', 'colours', or a CSV path

x, y, names = datasets.load(DATA)
if DATA != 'colours':
    x = datasets.standardise(x)    # attributes on different scales would otherwise dominate the distance
print(f'{x.shape[0]} samples, {x.shape[1]} attributes, {len(set(y)) if y is not None else 0} classes')

## 2. Train

`GeodesicSOM(10)` has 10·10² + 2 = 1002 neurons. Batch training is the default; `mode='online'` presents one
sample at a time.

In [ ]:
FREQ, EPOCHS = 10, 30

som = GeodesicSOM(FREQ, seed=3)
som.initialise(dataset=x)                    # linear (PCA) initialisation
som.train(x, epochs=EPOCHS, mode='batch')
print(f'{som}: {len(x)} samples, QE {som.quantisation_error(x):.3f}, TE {som.topographic_error(x):.3f}')

## 3. The viewer

Layers: *Neighbour distance*, *U-matrix* (per neuron), *Component* (one attribute's value), *Component
difference*, *PCA colour* (similar neurons get similar colours), *Hits* and *Classes*.

> **Live or static?** With `ipympl` installed the viewers below are live: drag a sphere to rotate it, click a
> neuron to inspect it, and use the radio buttons, check boxes and keys on the figure itself. Without it they are
> images, and the controls under each viewer redraw them.

In [ ]:
viewer = SOMViewer(som, x, labels=y, feature_names=names, projection='Equal Earth')

### Controls

In [ ]:
nb_setup.controls(viewer, neuron=int(np.argmax(viewer.hits)))

## 4. The same, from code

Everything the controls do is a method call, so a script can set up a view and save it.

In [ ]:
viewer.set_layer('Classes')
viewer.select_node(int(np.argmax(viewer.hits)))
viewer.set_view(30, -60)
print('layers:', viewer.layers)
print('centre of view:', tuple(round(v, 1) for v in viewer.centre))
if not WIDGET:                    # live figures have already changed above
    nb_setup.show(viewer)

### Try it: train your own

Pick the data, the size of the sphere and the training, then press **Run Interact**.

In [ ]:
@widgets.interact_manual(data=widgets.Dropdown(options=[*datasets.samples(), 'colours'], value='penguins'),
                         freq=widgets.IntSlider(value=8, min=2, max=20, description='frequency'),
                         epochs=widgets.IntSlider(value=30, min=1, max=100),
                         mode=widgets.ToggleButtons(options=['batch', 'online']),
                         standardise=widgets.Checkbox(value=True),
                         seed=widgets.IntSlider(value=3, min=0, max=20))
def train_sphere(data, freq, epochs, mode, standardise, seed):
    x, y, names = datasets.load(data)
    if standardise and data != 'colours':
        x = datasets.standardise(x)
    som = GeodesicSOM(freq, seed=seed)
    som.initialise(dataset=x)
    som.train(x, epochs=epochs, mode=mode)
    print(f'{som}: QE {som.quantisation_error(x):.3f}, TE {som.topographic_error(x):.3f}')
    with nb_setup.building():
        v = SOMViewer(som, x, labels=y, feature_names=names)
    nb_setup.show(v)
    nb_setup.controls(v, neuron=int(np.argmax(v.hits)))